In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('comet_pythia_1_4b.csv') # comet_v3.csv')

In [ ]:
cols_to_plot = [
    'sweep_pair.num_heads',
    'sweep_pair.num_mkeys',
    'sweep_pair.num_nkeys',
    'sweep_pair.dict_size',
    'explained_variance',
    'parameters_count',
    'accum_num_flops',
    'n_tokens',
    'performance/recovery_from_zero',
    'performance/recovery_from_mean'
]

In [ ]:
mapping = {
    'sweep id': ("1.4B", 100 * 1e6),
    'sweep id': ("1.4B", 500 * 1e6),
    'sweep id': ("1.4B", 1000 * 1e6),
    'sweep id': ("1.4B", 1000 * 1e6),
    
}

In [ ]:
kronsae = []
topk = []
for i, row in df.iterrows():
    if 'exp71' in row['Name'] and row['Sweep'] in mapping.keys():

        model_size, ref_tokens = mapping[row['Sweep']]

        entry = {
            'Name': row['Name'],
            'H': row['sweep_pair.num_heads'],
            'M': row['sweep_pair.num_mkeys'],
            'N': row['sweep_pair.num_nkeys'],
            'F': row['sweep_pair.dict_size'],
            'Explained Variance': row['explained_variance'],
            'CE_from_zero': row['performance/recovery_from_zero'],
            'CE_from_mean': row['performance/recovery_from_mean'],
            'Parameters': row['parameters_count'],
            'FLOPS': row['accum_num_flops'],
            'Tokens': row['n_tokens'],
            'Reference Tokens': ref_tokens,
            'Model size': model_size
        }
        
        if 'kronsae' in row['Name']:
            entry['Experts'] = entry['H'] * entry['M']
            kronsae.append(entry)
        else:
            topk.append(entry)

kronsae = pd.DataFrame(kronsae)
topk = pd.DataFrame(topk)

In [ ]:
model_size = '1.5B'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.lines import Line2D
import numpy as np
from scipy.interpolate import PchipInterpolator
# Set style
sns.set_style("whitegrid")
fig, axs = plt.subplots(1, 3, figsize=(15, 5), dpi=250, sharey=True) #, sharex=True, sharey=True)
#main_palette = "cividis"
main_palette = "colorblind"

import numpy as np

def pareto_frontier(xs, ys):
    """
    Extract the 2D Pareto frontier from points (xs, ys).
    Keeps points with strictly increasing y as x increases.
    Returns arrays (pf_x, pf_y) sorted by pf_x ascending.
    """
    pts = sorted(zip(xs, ys), key=lambda p: p[0])
    frontier = []
    best_y = -np.inf
    for x, y in pts:
        if y > best_y:
            frontier.append((x, y))
            best_y = y
    pf_x, pf_y = zip(*frontier)
    return np.array(pf_x), np.array(pf_y)

from scipy.optimize import curve_fit
from scipy.stats import linregress

def smooth_pareto_front(xs, ys, deg=2, num=200, scaler = 1.):
    """
    Compute and return a smooth polynomial approximation of the Pareto front.

    Args:
      xs (array-like):  x-coordinates of all points.
      ys (array-like):  y-coordinates of all points.
      deg (int):        degree of the fitting polynomial.
      num (int):        number of points to sample on the smooth curve.

    Returns:
      x_dense (np.ndarray):  sorted x-values along the front (length=num).
      y_smooth (np.ndarray): fitted y-values p(x_dense) (length=num).
    """
    # 1) Get the raw Pareto-optimal points
    print(xs, ys)
    #pf_x, pf_y = pareto_frontier(xs, ys)
    pf_x, pf_y = np.array(xs), np.array(ys)
    print(pf_x, pf_y)

    # 2) Fit a degree-deg polynomial via least squares
    #    p(x) = c[0]*x^deg + ... + c[deg]
    #coeffs = np.polyfit(pf_x, pf_y, deg=deg)     
    # def func(x, a, b, c):
    #     return a * x**2 + b * x + c  # a and d are redundant
    def func(x, a, b, c):
        return a * x**2 + b * x + c  # a and d are redundant
        
    popt, pcov = curve_fit(func, np.log(pf_x), pf_y)

    # 3) Evaluate on a dense grid
    x_dense = np.linspace(pf_x.min() - scaler * pf_x.min(), pf_x.max() + scaler* pf_x.max(), num)
    #y_smooth = np.polyval(coeffs, x_dense)  
    y_smooth = func(np.log(x_dense), *popt) #reg.predict(np.array(x_dense))

    return x_dense, y_smooth


for i, n_tokens in enumerate(sorted(kronsae['Reference Tokens'].unique(), reverse=False)):
    plot_df = kronsae[kronsae['Reference Tokens'] == n_tokens]
    plot_df = plot_df[plot_df["M"] <= plot_df["N"]]
    
    ax = axs[i]
    # Create plot with log scale for heads (powers of 2)
    sns.lineplot(
        data=plot_df,
        x='Parameters',  # Heads (powers of 2)
        y='Explained Variance',
        hue='F',
        style='M',
        markers=True,
        dashes=True,
        palette=main_palette,# 'tab10',
        markersize=8,
        linewidth=2,
        ax=ax,

        legend=i==0
    )

    # Set logarithmic x-axis with proper tick formatting
    #ax.set_xscale('log', base=2)
    # ax.set_xticks(plot_df['Parameters'].unique())
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

    # Customize plot
    # ax.set(
    #     xlabel='Trainable Parameters',
    #     ylabel='Explained Variance'
    # )
    ax.set_xlabel('Trainable Parameters', fontsize=18)
    ax.set_ylabel('Explained Variance', fontsize=18)
    ax.grid(True, alpha=0.3)
    

    #ax.set_title(f"Reference tokens: {n_tokens // 1e6:.0f}M")

    # plt.ylim(0.8, 0.875)

    #cplt = sns.color_palette('tab10', 3)
    cplt = sns.color_palette(main_palette, 3)
    g_results = sns.scatterplot(topk[topk['Reference Tokens'] == n_tokens], 
                    x = 'Parameters', 
                    y = 'Explained Variance', 
                    hue = 'F', 
                    s = 400, 
                    marker = '*', 
                    palette=cplt, 
                    legend = False, 
                    ax=ax)


    baseline = topk[topk['Reference Tokens'] == n_tokens]
    xs = baseline['Parameters'].values
    ys = baseline['Explained Variance'].values
    #print(xs, ys)
    

    # 3b) Fit a smooth Pareto front of degree 2
    x_front, y_front = smooth_pareto_front(xs, ys, num=200, scaler=1.9)

    # 3c) Draw the fitted curve
    ax.plot(
        x_front, y_front,
        color='black',
        lw=1.1,
        alpha=0.7,
        label='Fitted Pareto front' if i==0 else None
    )

    # baseline = plot_df[plot_df['Reference Tokens'] == n_tokens]
    # xs = baseline['Parameters'].values
    # ys = baseline['Explained Variance'].values
    idx = [grp['Explained Variance'].idxmax() for D_val, grp in plot_df.groupby("F")]
    xs1 = [grp['Parameters'][idx[j]] for j, (D_val, grp) in enumerate(plot_df.groupby("F"))]
    ys1 = [grp['Explained Variance'][idx[j]] for j, (D_val, grp) in enumerate(plot_df.groupby("F"))]
 

    # 3b) Fit a smooth Pareto front of degree 2
    x_front, y_front = smooth_pareto_front(xs1, ys1,  num=200, scaler=1.9)

    all_xs = np.concatenate([baseline['Parameters'].values, plot_df['Parameters'].values])

    ax.set_xlim(all_xs.min() - 0.15 * all_xs.min(), all_xs.max() + 0.15 * all_xs.max())

    # 3c) Draw the fitted curve
    ax.plot(
        x_front, y_front,
        color='red',
        lw=1.1,
        alpha=0.7,

        label='Fitted Pareto front' if i==0 else None
    )
    ax.set_title(f"Token Budget: {n_tokens // 1e6:.0f}M", fontsize=18)
    
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xscale("log", base=2)
    ax.set_ylim(0.65, 0.84)

    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.tick_params(axis='both', which='minor', labelsize=10)
    

axs[0].get_legend().remove()
handles, labels = axs[0].get_legend_handles_labels()
baseline_handles = [
    Line2D([0], [0], 
           marker="*", 
           #color="green", 
           color = "black",
           linestyle="",
           markersize=12, label=f"TopK SAE"),
]
axs[0].legend(
    #handles=handles[1:4] + handles[5:],  # Skip the first element (title)
    handles=handles[1:4] + handles[5:7] + baseline_handles, #+ baseline_handles,  # Skip the first element (title)
    #labels=['D = 32768', 'D = 65536', 'D = 131072', 'M = 2', 'M = 4', 'M=8'],# 'TopK'],
    labels=['$F = 2^{15}$', '$F = 2^{16}$', '$F = 2^{17}$', 'm = 2', 'm = 4',  'TopK'],fontsize=14,
    loc='upper left',
)
axs[0].set_xlabel('')
axs[2].set_xlabel('')

plt.tight_layout()
plt.savefig(f'parameters vs explained variance pythia1.5b.pdf')
plt.show()